In [ ]:
import duckdb
import pandas as pd
import numpy as np

# 1. Setup Dummy Data (Matching your schema)
data = {
    'person_id': ['A', 'A', 'A', 'A', 'B', 'B'],
    'month_reference': pd.to_datetime(['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-01-01', '2023-02-01']),
    'date_overdue': pd.to_datetime(['2023-01-10', '2023-02-10', '2023-03-10', '2023-04-10', '2023-01-15', '2023-02-15']).date,
    'date_paid': ['2023-01-08', '2023-02-12', None, '2023-04-09', '2023-01-15', '2023-02-20'], # String format as requested
    'invoice_value': [100.0, 200.0, 300.0, 400.0, 50.0, None] # One null for imputation test
}

# Create DataFrame with a subset of columns for readability
df = pd.DataFrame(data)

# 2. Define the SQL Query (DuckDB Syntax)
# Note: duckdb.query() can read the 'df' variable directly from the python environment
sql_query = """
WITH stage_1_cleaning AS (
    SELECT
        *,
        -- 1. Cast String to Date
        CAST(date_paid AS DATE) AS date_paid_cleaned,
        
        -- 2. Imputation
        COALESCE(invoice_value, 0.0) AS invoice_value_cleaned
    FROM
        df 
),

stage_2_features AS (
    SELECT
        *,
        -- 3. Calculate Days to Pay (DuckDB syntax: date_diff('part', start, end))
        date_diff('day', date_overdue, date_paid_cleaned) AS days_to_pay,

        -- 4. Rolling 3-Month Average
        AVG(invoice_value_cleaned) OVER (
            PARTITION BY person_id 
            ORDER BY month_reference 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rolling_avg_3m_invoice
    FROM
        stage_1_cleaning
)

SELECT 
    person_id, 
    month_reference, 
    invoice_value, 
    rolling_avg_3m_invoice, 
    days_to_pay 
FROM stage_2_features
ORDER BY person_id, month_reference;
"""

# 3. Execute
result_df = duckdb.query(sql_query).to_df()

# 4. Display
print(result_df)

  cd_pessoa mes_referencia  valor_fatura  rolling_avg_3m_fatura  days_to_pay
0         A     2023-01-01         100.0                  100.0           -2
1         A     2023-02-01         200.0                  150.0            2
2         A     2023-03-01         300.0                  200.0         <NA>
3         A     2023-04-01         400.0                  300.0           -1
4         B     2023-01-01          50.0                   50.0            0
5         B     2023-02-01           NaN                   25.0            5
